# Metadata Agent Evaluation with Strands Framework

This notebook evaluates the Metadata Generation Agent using the AWS Strands evaluation framework.

## AWS Credentials Setup

The metadata agent needs AWS credentials to access:
- **Amazon Bedrock** - For Claude model inference (`global.anthropic.claude-opus-4-6-v1`)
- **AWS Athena / Glue** - For retrieving table schemas and sampling data (`get_table_schema`, `sample_table_data`)
- **AWS Glue Data Catalog** - For writing AI-generated descriptions (`update_glue_table_metadata`)
- **Amazon S3** - For saving markdown metadata documents (`save_metadata_document_to_s3`)
- **Amazon DynamoDB** - For recording enrichment progress (`update_progress`)

### Credential Injection

This notebook uses an AWS profile configured locally. The boto3 session is created with this profile and then injected into the metadata agent using `set_boto_session()`. This ensures all AWS service calls within the agent use the same credentials.

### Agent Architecture

The metadata agent uses a **single-phase enrichment approach**:
1. For each table (one at a time):
   - `get_table_schema` — retrieve column definitions via Athena DESCRIBE TABLE (Glue fallback for DynamoDB-backed tables)
   - `sample_table_data` — fetch example rows to enrich descriptions with real values
   - `update_glue_table_metadata` — write table + column descriptions back to Glue
   - `save_metadata_document_to_s3` — persist a markdown document to S3 for Bedrock KB ingestion
   - `update_progress` — record job progress in DynamoDB

### Required Environment Variables
- `ARTIFACTS_BUCKET` — S3 bucket for metadata document storage
- `ATHENA_WORKGROUP` — Athena workgroup for schema and sample queries
- `ONTOLOGY_METADATA_TABLE` — DynamoDB table for job progress tracking
- `SEMANTIC_RAG_KB_ID` — Bedrock KB ID (used post-enrichment to trigger ingestion)
- `SEMANTIC_RAG_DATA_SOURCE_ID` — Bedrock KB data source ID
- `AWS_REGION` — AWS region (default: `us-east-1`)

In [1]:
# Install required packages for evaluation
# !pip install strands-agents strands-agents-tools strands-agents-evals pandas --quiet

In [2]:
# Verify installations
try:
    import strands
    import strands_evals
    print("✓ strands-agents installed:", strands.__version__ if hasattr(strands, '__version__') else "version unknown")
    print("✓ strands-evals installed:", strands_evals.__version__ if hasattr(strands_evals, '__version__') else "version unknown")
except ImportError as e:
    print(f"✗ Import error: {e}")

✓ strands-agents installed: version unknown
✓ strands-evals installed: version unknown


In [3]:
# Import required libraries
import sys
import os
import json
import time
import uuid
import pandas as pd
import boto3
from botocore.config import Config
from datetime import datetime, timezone

# Add agents directory to path
sys.path.append('../agents')

# Import agent functions
from metadata_agent.main import invoke, set_boto_session

print("✓ Imports successful")

✓ Imports successful


In [4]:
# Load environment variables from .env file (sourced from cdk/cdk-outputs-agentcore.json)
from dotenv import load_dotenv, set_key

ENV_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd(), '.env')
load_dotenv(dotenv_path=ENV_FILE, override=False)

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

# Create boto3 session early — needed for CloudFormation lookups below
config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')
session = boto3.Session(profile_name=os.environ.get('AWS_PROFILE', 'default'))
region = session.region_name or 'us-east-1'

# ── CloudFormation output resolver ──────────────────────────────────────────
# For any env var still set to 'XXX' (no .env or .env missing the key),
# resolve the value from live CloudFormation stack outputs.

def _get_stack_outputs(stack_name: str) -> dict:
    """Return {OutputKey: OutputValue} for a CloudFormation stack, or {} if not found."""
    cfn = session.client('cloudformation', region_name=region)
    try:
        resp = cfn.describe_stacks(StackName=stack_name)
        return {o['OutputKey']: o['OutputValue'] for o in resp['Stacks'][0].get('Outputs', [])}
    except Exception:
        return {}

# Map: env_var_name → (stack_name, output_key)
_CFN_MAP = {
    'ARTIFACTS_BUCKET':          (f'{PROJECT_NAME}-data-lake',    'ArtifactsBucketName'),
    'ATHENA_WORKGROUP':          (f'{PROJECT_NAME}-athena',       'AthenaWorkgroupName'),
    'ONTOLOGY_METADATA_TABLE':   (f'{PROJECT_NAME}-dynamodb',     'MetadataTableName'),
    'SEMANTIC_RAG_KB_ID':        (f'{PROJECT_NAME}-bedrock-kb',   'SemanticRagKbId'),
    'SEMANTIC_RAG_DATA_SOURCE_ID': (f'{PROJECT_NAME}-bedrock-kb', 'SemanticRagDataSourceId'),
    'KNOWLEDGE_BASE_ID':         (f'{PROJECT_NAME}-bedrock-kb',   'OntologyPatternsKbId'),
}

# Resolve: keep .env value if set and not 'XXX', otherwise query CloudFormation
_cfn_cache = {}  # stack_name → {OutputKey: OutputValue}
_resolved_from_cfn = []

for env_var, (stack, output_key) in _CFN_MAP.items():
    current = os.environ.get(env_var, 'XXX')
    if current != 'XXX':
        os.environ[env_var] = current
        continue
    # Lazy-load stack outputs
    if stack not in _cfn_cache:
        _cfn_cache[stack] = _get_stack_outputs(stack)
    value = _cfn_cache[stack].get(output_key, 'XXX')
    os.environ[env_var] = value
    if value != 'XXX':
        _resolved_from_cfn.append(env_var)

# ── Persist CFN-resolved values back to .env ────────────────────────────────
# So that downstream notebooks (e.g. 2_metadata_query_agent_evaluation) can
# read them without needing their own CloudFormation resolver.
if _resolved_from_cfn:
    for env_var in _resolved_from_cfn:
        set_key(ENV_FILE, env_var, os.environ[env_var])
    print(f"✓ Wrote {len(_resolved_from_cfn)} CFN-resolved variable(s) to .env: {', '.join(_resolved_from_cfn)}")

print("\n✓ Environment variables configured:")
for key in ['ARTIFACTS_BUCKET', 'ATHENA_WORKGROUP', 'ONTOLOGY_METADATA_TABLE',
            'SEMANTIC_RAG_KB_ID', 'SEMANTIC_RAG_DATA_SOURCE_ID', 'KNOWLEDGE_BASE_ID']:
    source = '(from CloudFormation → .env)' if key in _resolved_from_cfn else '(from .env)'
    val = os.environ.get(key, '(not set)')
    print(f"  {key} = {val}  {source}")

✓ Wrote 3 CFN-resolved variable(s) to .env: SEMANTIC_RAG_KB_ID, SEMANTIC_RAG_DATA_SOURCE_ID, KNOWLEDGE_BASE_ID

✓ Environment variables configured:
  ARTIFACTS_BUCKET = semantic-layer-artifacts-381492284087  (from .env)
  ATHENA_WORKGROUP = semantic-layer-workgroup  (from .env)
  ONTOLOGY_METADATA_TABLE = semantic-layer-metadata  (from .env)
  SEMANTIC_RAG_KB_ID = CUCH5JMBVX  (from CloudFormation → .env)
  SEMANTIC_RAG_DATA_SOURCE_ID = MEWZDA5QMN  (from CloudFormation → .env)
  KNOWLEDGE_BASE_ID = DJLRG31LYW  (from CloudFormation → .env)


In [5]:
# Verify credentials (session created in previous cell)
sts = session.client('sts', region_name=region, config=config)
try:
    identity = sts.get_caller_identity()
    print(f"✓ AWS Credentials Verified:")
    print(f"  Profile: {os.environ.get('AWS_PROFILE', 'default')}")
    print(f"  Account: ...{identity['Account'][-4:]}")
    print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
    print(f"  Region: {region}")
except Exception as e:
    print(f"✗ Failed to verify AWS credentials: {str(e)}")
    raise

# Inject session into metadata agent
set_boto_session(session)
print(f"\n✓ Agent configured to use boto3 session credentials")
print(f"  All AWS service calls (Bedrock, Athena, Glue, S3, DynamoDB) will use this session")

✓ AWS Credentials Verified:
  Profile: huthmac-demo
  Account: ...4087
  User/Role: huthmac-Isengard
  Region: us-east-1
2026-03-22 16:08:58,213 - metadata_agent.main - INFO - [MainThread] Boto3 session set with region: us-east-1

✓ Agent configured to use boto3 session credentials
  All AWS service calls (Bedrock, Athena, Glue, S3, DynamoDB) will use this session


## Define Evaluation Test Cases

Test cases target **S3 Tables (Iceberg)** in the S3 Tables namespace.

Each test case includes:
- **id**: Unique test identifier
- **category**: `single_table` or `multi_table`
- **tables**: List of `{databaseName, tableName, catalogId}` dicts (mirrors the production payload)
- **expected_outcomes**: Expected results to validate
- **validation_criteria**: Specific checks to perform

In [6]:
# S3 Tables (Iceberg) data source config
# CatalogId format: s3tablescatalog/<bucket-name>  (e.g. s3tablescatalog/semantic-layer-analytics-tables)
# Namespace: e.g. normalized

# Resolve catalog/database vars from CloudFormation if still 'XXX'
_CATALOG_CFN_MAP = {
    'S3T_DATABASE':    (f'{PROJECT_NAME}-data-lake',     'Namespace'),
    'DYNAMODB_CATALOG':  (f'{PROJECT_NAME}-athena',      'DynamoDBCatalogName'),
    'DYNAMODB_DATABASE': (f'{PROJECT_NAME}-glue-catalog', 'DynamoDBDatabaseName'),
}

for env_var, (stack, output_key) in _CATALOG_CFN_MAP.items():
    current = os.environ.get(env_var, 'XXX')
    if current != 'XXX':
        continue
    if stack not in _cfn_cache:
        _cfn_cache[stack] = _get_stack_outputs(stack)
    value = _cfn_cache[stack].get(output_key, 'XXX')
    os.environ[env_var] = value
    if value != 'XXX':
        _resolved_from_cfn.append(env_var)

# S3T_CATALOG is special: s3tablescatalog/<bucket-name>
# The prefix comes from CFN, the bucket name is {PROJECT_NAME}-analytics-tables
S3T_CATALOG = os.environ.get('S3T_CATALOG', 'XXX')
if S3T_CATALOG == 'XXX':
    if f'{PROJECT_NAME}-data-lake' not in _cfn_cache:
        _cfn_cache[f'{PROJECT_NAME}-data-lake'] = _get_stack_outputs(f'{PROJECT_NAME}-data-lake')
    prefix = _cfn_cache[f'{PROJECT_NAME}-data-lake'].get('S3TablesFederatedCatalogName', 's3tablescatalog')
    S3T_CATALOG = f"{prefix}/{PROJECT_NAME}-analytics-tables"
    os.environ['S3T_CATALOG'] = S3T_CATALOG
    _resolved_from_cfn.append('S3T_CATALOG')

S3T_DATABASE      = os.environ.get('S3T_DATABASE', 'XXX')
DYNAMODB_CATALOG  = os.environ.get('DYNAMODB_CATALOG', 'XXX')
DYNAMODB_DATABASE = os.environ.get('DYNAMODB_DATABASE', 'XXX')

# Persist any newly CFN-resolved catalog/database vars to .env
_catalog_resolved = [k for k in ['S3T_CATALOG', 'S3T_DATABASE', 'DYNAMODB_CATALOG', 'DYNAMODB_DATABASE']
                     if k in _resolved_from_cfn]
if _catalog_resolved:
    for env_var in _catalog_resolved:
        set_key(ENV_FILE, env_var, os.environ[env_var])
    print(f"✓ Wrote {len(_catalog_resolved)} catalog variable(s) to .env: {', '.join(_catalog_resolved)}")

print("\n✓ Catalog/database configured:")
for key, val in [('S3T_CATALOG', S3T_CATALOG), ('S3T_DATABASE', S3T_DATABASE),
                 ('DYNAMODB_CATALOG', DYNAMODB_CATALOG), ('DYNAMODB_DATABASE', DYNAMODB_DATABASE)]:
    source = '(from CloudFormation → .env)' if key in _resolved_from_cfn else '(from .env)'
    print(f"  {key} = {val}  {source}")

def s3t(table_name: str) -> dict:
    return {
        'databaseName': S3T_DATABASE,
        'tableName':    table_name,
        'catalogId':    S3T_CATALOG,
        'dataSource':   'AwsDataCatalog',
        'tableId':      f'{S3T_CATALOG}::{S3T_DATABASE}.{table_name}',
    }

def dynamot(table_name: str) -> dict:
    return {
        'databaseName': DYNAMODB_DATABASE,
        'tableName':    table_name,
        'catalogId':    DYNAMODB_CATALOG,
        'dataSource':   DYNAMODB_CATALOG,
        'tableId':      f'{DYNAMODB_CATALOG}::{DYNAMODB_DATABASE}.{table_name}',
    }

test_cases = [
    {
        'id': 'single_table_basic',
        'ontology_id': 'semantic-rag-test-001',
        'category': 'single_s3table',
        'dynamodb_config': {
            'name': 'admin-codes-s3tables',
            'type': 'SemanticRAG',
            'dataSourcesDescription': 'Iceberg S3 Tables',
            'useCasesDescription': 'Enable business users to query administrative codes',
            'dataSources': [s3t('admin_codes')],
        },
        'expected_outcomes': {
            'table_created': ['admin_codes'],
            'properties_per_table': 'all_columns',
            'traceability_mappings': True,
        },
        'validation_criteria': [
            'Table description explains business entity',
            'Column descriptions use business language (not raw column names)',
            'S3 markdown document saved successfully',
            'Markdown contains Overview, Columns, and Sample Data sections',
            'Column comment length <= 255 chars',
        ],
    },
    # {
    #     'id': 'single_table_basic',
    #     'ontology_id': 'rag-test-002',
    #     'category': 'single_dynamodb_table',
    #     'dynamodb_config': {
    #         'name': 'admin-codes-dyndb',
    #         'type': 'SemanticRAG',
    #         'dataSourcesDescription': 'Direct DynamoDB Table',
    #         'useCasesDescription': 'Enable business users to query administrative codes',
    #         'dataSources': [dynamot('semantic-layer-admin-codes')],
    #     },
    #     'expected_outcomes': {
    #         'table_created': ['semantic-layer-admin-codes'],
    #         'properties_per_table': 'all_columns',
    #         'traceability_mappings': True,
    #     },
    #     'validation_criteria': [
    #         'Table description explains business entity',
    #         'Column descriptions use business language (not raw column names)',
    #         'S3 markdown document saved successfully',
    #         'Markdown contains Overview, Columns, and Sample Data sections',
    #         'Column comment length <= 255 chars',
    #     ],
    # },
]

print(f"\n✓ Defined {len(test_cases)} test cases")
print(f"\nCategories:")
categories = {}
for tc in test_cases:
    cat = tc['category']
    categories[cat] = categories.get(cat, 0) + 1
for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count} test case(s)")
print(f"\nTables used:")
for tc in test_cases:
    tables = [ds['tableName'] for ds in tc['dynamodb_config']['dataSources']]
    print(f"  {tc['id']}: {', '.join(tables)}")


✓ Catalog/database configured:
  S3T_CATALOG = s3tablescatalog/semantic-layer-analytics-tables  (from .env)
  S3T_DATABASE = normalized  (from .env)
  DYNAMODB_CATALOG = dynamodb_catalog  (from .env)
  DYNAMODB_DATABASE = default  (from .env)

✓ Defined 1 test cases

Categories:
  single_s3table: 1 test case(s)

Tables used:
  single_table_basic: admin_codes


In [7]:
# helper method to create entry in semantic-metadata-layer table, replicating UI application logic

METADATA_TABLE = os.environ.get('ONTOLOGY_METADATA_TABLE', 'semantic-layer-metadata')

def seed_test_ontology(
    ontology_id: str,
    tables: list,
    name: str = 'semantic-rag-eval-test',
    data_sources_description: str = '',
    use_cases_description: str = '',
    created_by: str = 'eval@semantic-layer.local',
) -> None:
    """
    Write a DynamoDB item matching the shape created by the admin flow.
    Each entry in `tables` is a full data-source dict from s3t().
    """
    dynamodb = session.resource('dynamodb')
    ddb_table = dynamodb.Table(METADATA_TABLE)
    now = datetime.now(timezone.utc).isoformat()
    ts = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
    name_with_ts = f"{name}-{ts}"
    data_sources = [
        {
            'databaseName': t['databaseName'],
            'tableName':    t['tableName'],
            'catalogId':    t.get('catalogId', 'AWSDataCatalog'),
            'dataSource':   t.get('dataSource', 'AwsDataCatalog'),
            'tableId':      t.get('tableId', f"{t.get('catalogId', 'AWSDataCatalog')}::{t['databaseName']}.{t['tableName']}"),
        }
        for t in tables
    ]

    item = {
        'id':             ontology_id,
        'version':        'v1',
        'type':           'SemanticRAG',
        'name':           name_with_ts,
        'status':         'pending',
        'configuration':  {},
        'dataSources':    data_sources,
        'createdAt':      now,
        'updatedAt':      now,
        'buildStartedAt': now,
        'createdBy':      created_by,
    }
    if data_sources_description:
        item['dataSourcesDescription'] = data_sources_description
    if use_cases_description:
        item['useCasesDescription'] = use_cases_description

    ddb_table.put_item(Item=item)
    print(f"✓ Seeded DynamoDB item: {ontology_id} → name='{name_with_ts}' ({len(data_sources)} table(s))")

print("✓ seed_test_ontology defined")

✓ seed_test_ontology defined


## Setup Strands Evaluation Framework

Configure telemetry and evaluation infrastructure for capturing agent execution traces.

In [8]:
from strands_evals import Case, Experiment
from strands_evals.evaluators import (
    HelpfulnessEvaluator,
    FaithfulnessEvaluator,
    ToolSelectionAccuracyEvaluator,
    GoalSuccessRateEvaluator,
)
from strands_evals.mappers import StrandsInMemorySessionMapper
from strands_evals.telemetry import StrandsEvalsTelemetry

# Setup telemetry for capturing agent spans
telemetry = StrandsEvalsTelemetry().setup_in_memory_exporter()
memory_exporter = telemetry.in_memory_exporter

print("✓ Strands evaluation framework initialized")
print(f"  Telemetry: In-memory exporter configured")
print(f"  Ready to capture agent traces")

2026-03-22 16:08:58,246 - strands_evals.telemetry.config - INFO - [MainThread] Initializing tracer for strands-evals
2026-03-22 16:08:58,246 - strands_evals.telemetry.config - INFO - [MainThread] Enabling in-memory export for strands-evals
✓ Strands evaluation framework initialized
  Telemetry: In-memory exporter configured
  Ready to capture agent traces


## Convert Test Cases to Strands Format

Transform the test dataset into Strands `Case` objects with structured prompts that mirror the production invocation payload.

In [9]:
strands_cases = []

for test_case in test_cases:
    id = f"semantic-rag-{test_case['id']}-{uuid.uuid4().hex[:8]}"
    cfg = test_case['dynamodb_config']
    tables = cfg['dataSources']

    table_names = ', '.join(t['tableName'] for t in tables)
    prompt = (
        f"Enrich the Glue Data Catalog and Knowledge Base for "
        f"{len(tables)} table(s): {table_names}. Job ID: {id}."
    )
    expected_output = (
        f"Enriched metadata documents for: {table_names}. "
        f"Each table has a business-friendly description, per-column descriptions, "
        f"and a markdown document saved to S3."
    )

    case = Case[str, str](
        name=test_case['id'],
        input=prompt,
        expected_output=expected_output,
        metadata={
            'category': test_case['category'],
            'id': id,
            'tables': tables,
            'dynamodb_config': {
                'name': cfg.get('name', f"semantic-rag-{test_case['id']}"),
                'dataSourcesDescription': cfg.get('dataSourcesDescription', f"Tables for semantic enrichment: {table_names}"),
                'useCasesDescription': cfg.get('useCasesDescription', 'Enable business users to query enriched metadata via Bedrock Knowledge Base'),
            },
            'expected_outcomes': test_case['expected_outcomes'],
            'validation_criteria': test_case['validation_criteria'],
        }
    )
    strands_cases.append(case)

print(f"✓ Converted {len(strands_cases)} test case(s) to Strands Case format")
print(f"\nCases:")
for case in strands_cases:
    tables = [t['tableName'] for t in case.metadata['tables']]
    print(f"  {case.name} ({case.metadata['category']}): {', '.join(tables)}")

metadata_id = id
%store metadata_id
print(f"\n✓ metadata_id stored: {metadata_id}")

✓ Converted 1 test case(s) to Strands Case format

Cases:
  single_table_basic (single_s3table): admin_codes
Stored 'metadata_id' (str)

✓ metadata_id stored: semantic-rag-single_table_basic-8f7bcf77


## Define Task Function

Create the task function that seeds a DynamoDB record, calls `invoke()` (the full production entry point), then polls DynamoDB until the background thread signals `completed` or `failed`.

In [10]:
def metadata_agent_task(case: Case) -> dict:
    memory_exporter.clear()
    tables = case.metadata['tables']
    id = case.metadata['id']
    cfg = case.metadata.get('dynamodb_config', {})
    annotations = case.metadata.get('annotations', [])

    # Seed DynamoDB — mirrors what the admin flow creates before invoking the agent
    seed_test_ontology(
        id,
        tables,
        name=cfg.get('name', f"semantic-rag-{id[:8]}"),
        data_sources_description=cfg.get('dataSourcesDescription', ''),
        use_cases_description=cfg.get('useCasesDescription', ''),
    )

    try:
        start_time = datetime.now()
        failures = []

        # Invoke the metadata agent — same payload Lambda sends via AgentCore: {"id": <id>}.
        # invoke() reads DynamoDB (including enrichmentAnnotations written there by the service
        # layer), updates status to 'processing', and starts the background thread.
        ann_label = f" with {len(annotations)} annotation(s)" if annotations else ""
        print(f"[{case.name}] Invoking metadata agent for: {id}{ann_label}")
        invoke(payload={"id": id}, context={})

        # Poll DynamoDB until the background thread finishes
        dynamodb_r = session.resource('dynamodb')
        ddb_table = dynamodb_r.Table(METADATA_TABLE)
        max_wait_secs = 600   # 10 minutes
        poll_interval = 15    # seconds

        print(f"[{case.name}] Polling DynamoDB for completion (max {max_wait_secs}s)...")
        final_item = {}
        final_status = 'processing'
        while (datetime.now() - start_time).total_seconds() < max_wait_secs:
            time.sleep(poll_interval)
            resp = ddb_table.get_item(Key={'id': id, 'version': 'v1'})
            final_item = resp.get('Item', {})
            final_status = final_item.get('status', 'unknown')
            elapsed = (datetime.now() - start_time).total_seconds()
            print(f"  [{elapsed:.0f}s] status={final_status}")
            if final_status in ('completed', 'failed'):
                break

        duration = (datetime.now() - start_time).total_seconds()

        # Build summary from final DynamoDB state
        if final_status == 'completed':
            summary = final_item.get('summary', f"Processed {len(tables)}/{len(tables)} tables.")
        elif final_status == 'failed':
            error_msg = final_item.get('error', 'unknown error')
            failures.append(error_msg)
            summary = f"Metadata enrichment failed: {error_msg}"
        else:
            failures.append(f"Timed out after {max_wait_secs}s with status: {final_status}")
            summary = f"Metadata enrichment timed out. Final status: {final_status}"

        print(f"✓ {summary}")

        finished_spans = list(memory_exporter.get_finished_spans())
        mapper = StrandsInMemorySessionMapper()
        session_obj = mapper.map_to_session(finished_spans, session_id=case.session_id)

        return {
            'output':     summary,
            'trajectory': session_obj,
            'duration':   duration,
            'success':    len(failures) == 0,
            'error':      ', '.join(failures) if failures else None,
        }

    except Exception as e:
        print(f"✗ Error in {case.name}: {str(e)}")
        import traceback; traceback.print_exc()
        return {'output': '', 'trajectory': None, 'duration': 0, 'success': False, 'error': str(e)}

print("✓ metadata_agent_task defined")

✓ metadata_agent_task defined


## Create Built-in Evaluators

Configure Strands evaluators for multi-dimensional assessment of metadata enrichment quality.

In [11]:
from strands.models import BedrockModel

# Create judge model using the same boto session
judge_model = BedrockModel(
    model_id='global.anthropic.claude-sonnet-4-6',
    temperature=0.0,
    boto_session=session,
)

# Custom system prompt for ToolSelectionAccuracyEvaluator.
# The default prompt evaluates each tool call in isolation, which causes
# the judge to lose context about where in the workflow the call occurs
# (especially for the final update_progress call).  We inject the expected
# tool ordering so the judge can reason about sequence position.
TOOL_SELECTION_SYSTEM_PROMPT = """You are an objective judge evaluating if an AI assistant's action is justified at this specific point in the conversation.

## Expected Workflow (tool ordering reference)
The metadata enrichment agent is expected to call tools in roughly this order for each table:
1. `get_single_table_schema` — retrieve column definitions (always first)
2. `sample_table_data` — fetch example rows to ground descriptions (may retry on timeout)
3. `retrieve_ontology_patterns` — optional; retrieve domain vocabulary from knowledge base
4. `update_glue_table_metadata` — write table + column descriptions back to Glue
5. `save_metadata_document_to_s3` — persist a markdown document to S3
6. `update_progress` — record job progress in DynamoDB (always last, after all processing is done)

Use this ordering as context when judging whether a tool call is appropriate at its position
in the conversation. A tool call that appears at the expected position in this workflow
should be considered justified, even if earlier tool calls are not shown in the conversation
history (they may have been executed in the same agent turn).

## Evaluation Question:
Given the current state of the conversation AND the expected workflow above, is the Agent
justified in calling this specific action at this point?

Consider:
1. Does this action reasonably address the user's current request or implied need?
2. Is the action aligned with the user's expressed or implied intent?
3. Are the minimum necessary parameters available to make the call useful?
4. Would a helpful assistant reasonably take this action to serve the user?
5. Is this action consistent with the expected workflow ordering?

## Evaluation Guidelines:
- Be practical and user-focused — actions that help the user achieve their goals are justified
- Consider implied requests and contextual clues when evaluating action appropriateness
- If an action has sufficient required parameters to be useful (even if not optimal), it may be acceptable
- If an action reasonably advances the conversation toward fulfilling the user's needs, consider it valid
- If multiple actions could work, but this one is reasonable, consider it justified
- `update_progress` called after schema/sample/save steps is the EXPECTED final step — do not penalise it

## Output Format:
First, provide a brief analysis of why this action is or is not justified at this point in the conversation.
Then, answer the evaluation question with EXACTLY ONE of these responses:
- Yes (if the action reasonably serves the user's intention at this point)
- No (if the action clearly does not serve the user's intention at this point)"""

# Initialize built-in evaluators
evaluators = [
    # TRACE_LEVEL: Description quality — are the generated descriptions helpful?
    HelpfulnessEvaluator(model=judge_model),

    # TRACE_LEVEL: Faithfulness — do descriptions accurately reflect the schema/sample data?
    FaithfulnessEvaluator(model=judge_model),

    # TOOL_LEVEL: Did the agent call the right tools in the right order?
    ToolSelectionAccuracyEvaluator(model=judge_model, system_prompt=TOOL_SELECTION_SYSTEM_PROMPT),

    # SESSION_LEVEL: Did the agent successfully complete the full enrichment workflow?
    GoalSuccessRateEvaluator(model=judge_model),
]

print(f"✓ Created {len(evaluators)} evaluators:")
print(f"  1. HelpfulnessEvaluator            - Description quality & business clarity")
print(f"  2. FaithfulnessEvaluator           - Schema accuracy & hallucination detection")
print(f"  3. ToolSelectionAccuracyEvaluator  - Correct tool usage & ordering (custom prompt w/ expected workflow)")
print(f"  4. GoalSuccessRateEvaluator        - End-to-end enrichment success")
print(f"\n  All evaluators use claude-sonnet-4-6 as judge")

✓ Created 4 evaluators:
  1. HelpfulnessEvaluator            - Description quality & business clarity
  2. FaithfulnessEvaluator           - Schema accuracy & hallucination detection
  3. ToolSelectionAccuracyEvaluator  - Correct tool usage & ordering (custom prompt w/ expected workflow)
  4. GoalSuccessRateEvaluator        - End-to-end enrichment success

  All evaluators use claude-sonnet-4-6 as judge


## Run Evaluation Experiment

Execute test cases and evaluate with all built-in evaluators.

In [12]:
# Create experiment with test cases and evaluators
experiment = Experiment[str, str](
    cases=strands_cases,
    evaluators=evaluators,
)

print(f"Starting experiment with {len(strands_cases)} test case(s)...")
print(f"{'='*70}\n")

# Run evaluations
reports = experiment.run_evaluations(metadata_agent_task)

print(f"\n{'='*70}")
print(f"✓ Evaluation complete!")
print(f"  Generated {len(reports)} evaluation report(s)")

Starting experiment with 1 test case(s)...

✓ Seeded DynamoDB item: semantic-rag-single_table_basic-8f7bcf77 → name='admin-codes-s3tables-20260322-200858' (1 table(s))
[single_table_basic] Invoking metadata agent for: semantic-rag-single_table_basic-8f7bcf77
2026-03-22 16:08:58,544 - metadata_agent.main - INFO - [MainThread] [Entrypoint] Starting metadata enrichment for: semantic-rag-single_table_basic-8f7bcf77
2026-03-22 16:08:58,877 - metadata_agent.main - INFO - [MainThread] DynamoDB status updated: semantic-rag-single_table_basic-8f7bcf77 (version=v1) → processing
2026-03-22 16:08:58,878 - metadata_agent.main - INFO - [MainThread] [Entrypoint] 1 tables to process


{"timestamp": "2026-03-22T20:08:58.879Z", "level": "INFO", "message": "Async task started: metadata_enrichment (ID: 2194142258593441646)", "logger": "bedrock_agentcore.app"}


2026-03-22 16:08:58,879 - bedrock_agentcore.app - INFO - [MainThread] Async task started: metadata_enrichment (ID: 2194142258593441646)
2026-03-22 16:08:58,880 - metadata_agent.main - INFO - [metadata-enrichment] [Table 1/1] normalized.admin_codes (catalog: s3tablescatalog/semantic-layer-analytics-tables)
[single_table_basic] Polling DynamoDB for completion (max 600s)...
2026-03-22 16:08:58,887 - strands.telemetry.metrics - INFO - [ThreadPoolExecutor-1_0] Creating Strands MetricsClient


I'll process the `admin_codes` table. Let me start by gathering schema and sample data.
Tool #1: get_single_table_schema

Tool #2: sample_table_data
2026-03-22 16:09:02,938 - metadata_agent.main - INFO - [asyncio_1] get_single_table_schema called: database='normalized', table='admin_codes', catalog_id='s3tablescatalog/semantic-layer-analytics-tables'
2026-03-22 16:09:02,939 - metadata_agent.main - INFO - [asyncio_1] Routing 'normalized.admin_codes': S3 Tables (Glue direct)
2026-03-22 16:09:02,939 - met

{"timestamp": "2026-03-22T20:15:38.696Z", "level": "INFO", "message": "Async task completed: metadata_enrichment (ID: 2194142258593441646, Duration: 399.82s)", "logger": "bedrock_agentcore.app"}


2026-03-22 16:15:38,696 - bedrock_agentcore.app - INFO - [metadata-enrichment] Async task completed: metadata_enrichment (ID: 2194142258593441646, Duration: 399.82s)
  [407s] status=completed
✓ Processed 1/1 tables.

✓ Evaluation complete!
  Generated 4 evaluation report(s)


## Aggregate Metrics

Analyze scores across all test cases and evaluators.

In [13]:
# Aggregate evaluation scores across all test cases
evaluation_data = []

n_cases = len(reports[0].cases) if reports else 0

for i in range(n_cases):
    case_dict = reports[0].cases[i]
    metadata = case_dict.get('metadata') or {}
    case_data = {
        'test_id': case_dict.get('name', f'case_{i}'),
        'category': metadata.get('category', 'unknown'),
        'tables': ', '.join(t['tableName'] for t in metadata.get('tables', [])),
    }
    for j, report in enumerate(reports):
        eval_name = evaluators[j].get_type_name()
        case_data[eval_name] = report.scores[i] if i < len(report.scores) else None
    evaluation_data.append(case_data)

df_evals = pd.DataFrame(evaluation_data)
evaluator_cols = [c for c in df_evals.columns if c not in ['test_id', 'category', 'tables']]

print("=== EVALUATION SCORES BY TEST CASE ===\n")
print(df_evals.to_string(index=False))

print("\n=== AVERAGE SCORES BY EVALUATOR ===\n")
print(df_evals[evaluator_cols].mean().to_string())

print("\n=== SCORES BY CATEGORY ===\n")
print(df_evals.groupby('category')[evaluator_cols].mean().to_string())

=== EVALUATION SCORES BY TEST CASE ===

           test_id       category      tables  HelpfulnessEvaluator  FaithfulnessEvaluator  ToolSelectionAccuracyEvaluator  GoalSuccessRateEvaluator
single_table_basic single_s3table admin_codes                   1.0                    1.0                             1.0                       1.0

=== AVERAGE SCORES BY EVALUATOR ===

HelpfulnessEvaluator              1.0
FaithfulnessEvaluator             1.0
ToolSelectionAccuracyEvaluator    1.0
GoalSuccessRateEvaluator          1.0

=== SCORES BY CATEGORY ===

                HelpfulnessEvaluator  FaithfulnessEvaluator  ToolSelectionAccuracyEvaluator  GoalSuccessRateEvaluator
category                                                                                                             
single_s3table                   1.0                    1.0                             1.0                       1.0


## Pass/Fail Analysis

Identify which test cases passed or failed each evaluator.

In [14]:
# Analyze pass/fail status for each evaluator
pass_fail_data = []

for i in range(n_cases):
    case_dict = reports[0].cases[i]
    metadata = case_dict.get('metadata') or {}
    case_pf = {
        'test_id': case_dict.get('name', f'case_{i}'),
        'category': metadata.get('category', 'unknown'),
    }
    for j, report in enumerate(reports):
        eval_name = evaluators[j].get_type_name()
        case_pf[eval_name] = report.test_passes[i] if i < len(report.test_passes) else None
    pass_fail_data.append(case_pf)

df_pass_fail = pd.DataFrame(pass_fail_data)
pf_cols = [c for c in df_pass_fail.columns if c not in ['test_id', 'category']]

print("=== PASS/FAIL BY TEST CASE ===\n")
print(df_pass_fail.to_string(index=False))

print("\n=== PASS RATE BY EVALUATOR ===\n")
print(df_pass_fail[pf_cols].mean().to_string())

print("\n=== PASS RATE BY CATEGORY ===\n")
print(df_pass_fail.groupby('category')[pf_cols].mean().to_string())

=== PASS/FAIL BY TEST CASE ===

           test_id       category  HelpfulnessEvaluator  FaithfulnessEvaluator  ToolSelectionAccuracyEvaluator  GoalSuccessRateEvaluator
single_table_basic single_s3table                  True                   True                            True                      True

=== PASS RATE BY EVALUATOR ===

HelpfulnessEvaluator              1.0
FaithfulnessEvaluator             1.0
ToolSelectionAccuracyEvaluator    1.0
GoalSuccessRateEvaluator          1.0

=== PASS RATE BY CATEGORY ===

                HelpfulnessEvaluator  FaithfulnessEvaluator  ToolSelectionAccuracyEvaluator  GoalSuccessRateEvaluator
category                                                                                                             
single_s3table                   1.0                    1.0                             1.0                       1.0


## Display Detailed Report

View comprehensive evaluation metrics for each evaluator.

In [15]:
if reports:
    for j, report in enumerate(reports):
        eval_name = evaluators[j].get_type_name()
        print(f"\n=== {eval_name} ===")
        report.display()
else:
    print("No reports generated")


=== HelpfulnessEvaluator ===


╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                         Test Case Results                         
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ index ┃ name               ┃ score ┃ test_pass ┃ reason ┃ input ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ ▶ 0   │ single_table_basic │ 1.00  │ ✅        │ ...    │ ...   │
└───────┴────────────────────┴───────┴───────────┴────────┴───────┘


=== FaithfulnessEvaluator ===


╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                         Test Case Results                         
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ index ┃ name               ┃ score ┃ test_pass ┃ reason ┃ input ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ ▶ 0   │ single_table_basic │ 1.00  │ ✅        │ ...    │ ...   │
└───────┴────────────────────┴───────┴───────────┴────────┴───────┘


=== GoalSuccessRateEvaluator ===


╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                         Test Case Results                         
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ index ┃ name               ┃ score ┃ test_pass ┃ reason ┃ input ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ ▶ 0   │ single_table_basic │ 1.00  │ ✅        │ ...    │ ...   │
└───────┴────────────────────┴───────┴───────────┴────────┴───────┘

## Save Evaluation Results

Export detailed evaluation data for further analysis.

In [16]:
import os as _os
_os.makedirs('../data/eval/results', exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# # Save scores DataFrame
# scores_file = f'../data/eval/results/metadata_evaluation_scores_{timestamp}.csv'
# df_evals.to_csv(scores_file, index=False)
# print(f"✓ Scores saved to: {scores_file}")

# # Save pass/fail DataFrame
# pass_fail_file = f'../data/eval/results/metadata_evaluation_pass_fail_{timestamp}.csv'
# df_pass_fail.to_csv(pass_fail_file, index=False)
# print(f"✓ Pass/fail data saved to: {pass_fail_file}")

# Save detailed JSON — one entry per test case with all evaluator results
detailed_results = []
for i in range(n_cases):
    case_dict = reports[0].cases[i]
    metadata = case_dict.get('metadata') or {}
    case_result = {
        'test_id': case_dict.get('name', f'case_{i}'),
        'category': metadata.get('category', 'unknown'),
        'tables': metadata.get('tables', []),
        'input': str(case_dict.get('input', ''))[:200] + '...',
        'expected': case_dict.get('expected_output', ''),
        'evaluations': {},
    }
    for j, report in enumerate(reports):
        eval_name = evaluators[j].get_type_name()
        raw_detailed = report.detailed_results[i] if i < len(report.detailed_results) else []
        eval_outputs = [
            {'score': r.score, 'test_pass': r.test_pass, 'label': r.label, 'reasoning': r.reason}
            for r in raw_detailed
        ] if isinstance(raw_detailed, list) else []
        case_result['evaluations'][eval_name] = {
            'score': report.scores[i] if i < len(report.scores) else None,
            'test_pass': report.test_passes[i] if i < len(report.test_passes) else None,
            'reason': report.reasons[i] if i < len(report.reasons) else '',
            'detailed': eval_outputs,
        }
    detailed_results.append(case_result)

detailed_file = f'../data/eval/results/metadata_evaluation_detailed_{timestamp}.json'
with open(detailed_file, 'w') as f:
    json.dump(detailed_results, f, indent=2)
print(f"✓ Detailed results saved to: {detailed_file}")

print(f"\n{'='*70}")
print(f"✓ All evaluation results saved")
# print(f"  Scores:   {scores_file}")
# print(f"  Pass/Fail: {pass_fail_file}")
print(f"  Detailed:  {detailed_file}")

✓ Detailed results saved to: ../data/eval/results/metadata_evaluation_detailed_20260322_161742.json

✓ All evaluation results saved
  Detailed:  ../data/eval/results/metadata_evaluation_detailed_20260322_161742.json


## Annotation Test

Re-enriches the S3 Tables config created by test case 1 with user-supplied annotation hints.

Replicates the exact production flow triggered by `POST /revise/{id}/{version_id}` (202):
1. Computes next version from existing DynamoDB records
2. Reads v1 (mutable pointer / current config)
3. Stamps v1 with `revisionMode=True`, `targetVersion`, `revisionInstructions`
4. Invokes the agent asynchronously; agent reads revision context from DynamoDB

In [17]:
# Retrieve the existing config ID seeded by test case 1 (S3 Tables admincode)
config_id = strands_cases[0].metadata['id']
print(f"Using existing config ID from test case 1: {config_id}")

annotations = [
    {
        "target": "admincodetype",
        "instruction": "Replace entire column description with THIS IS A REVISION TEST",
    },
    {
        "table": "admin_codes",
        "instruction": (
            "Replace entire description with THIS IS A REVISION TEST"
        ),
    },
]

# Mirror metadata_service.start_metadata_revision():
#   1. Compute next version from existing DynamoDB records
#   2. Get v1 record (mutable pointer / current config)
#   3. Stamp v1 with revisionMode=True, targetVersion, revisionInstructions
#   4. Persist so the agent picks it up when invoked below
from boto3.dynamodb.conditions import Key as DKey

ddb_resource = session.resource('dynamodb')
ddb_table = ddb_resource.Table(METADATA_TABLE)

version_resp = ddb_table.query(KeyConditionExpression=DKey('id').eq(config_id))
all_versions = version_resp.get('Items', [])
highest = max(
    (int(v['version'].lstrip('v')) for v in all_versions if v.get('version', '').startswith('v')),
    default=1,
)
next_version = f'v{highest + 1}'
base_version = 'v1'

resp = ddb_table.get_item(Key={'id': config_id, 'version': 'v1'})
current_config = resp.get('Item')
if not current_config:
    raise ValueError(f"Metadata config not found: {config_id}")

current_config.update({
    'revisionMode': True,
    'revisionBaseVersion': base_version,
    'revisionInstructions': annotations,
    'targetVersion': next_version,
    'status': 'pending',
    'updatedAt': datetime.now(timezone.utc).isoformat(),
})
ddb_table.put_item(Item=current_config)

print(f"✓ Stamped v1 record with revision context → targetVersion={next_version}, {len(annotations)} instruction(s)")

# Invoke the agent directly — same payload Lambda sends via AgentCore: {"id": <config_id>}.
# The agent reads revisionMode + revisionInstructions from DynamoDB and drives the revision build.
print(f"Invoking metadata agent for: {config_id} with {len(annotations)} annotation(s)")
invoke(payload={"id": config_id}, context={})

# Poll the target version record directly.
# After _write_versioned_completion, v1 is marked 'inactive' and the new version
# (next_version) is written as 'completed'. Polling v1 would only ever see 'inactive'.
start_time = datetime.now()
final_status = 'unknown'
final_item = {}
max_wait_secs = 600

print(f"Polling {next_version} for completion (max {max_wait_secs}s)...")
while (datetime.now() - start_time).total_seconds() < max_wait_secs:
    time.sleep(15)
    resp = ddb_table.get_item(Key={'id': config_id, 'version': next_version})
    final_item = resp.get('Item', {})
    final_status = final_item.get('status', 'unknown')
    elapsed = (datetime.now() - start_time).total_seconds()
    print(f"  [{elapsed:.0f}s] {next_version} status={final_status}")
    if final_status in ('completed', 'failed'):
        break

icon = '✓' if final_status == 'completed' else '✗'
print(f"\n{icon} Final status: {final_status}")
print(f"  version: {final_item.get('version', '')}")
print(f"  summary: {final_item.get('summary', '')}")

Using existing config ID from test case 1: semantic-rag-single_table_basic-8f7bcf77
✓ Stamped v1 record with revision context → targetVersion=v2, 2 instruction(s)
Invoking metadata agent for: semantic-rag-single_table_basic-8f7bcf77 with 2 annotation(s)
2026-03-22 16:17:42,585 - metadata_agent.main - INFO - [MainThread] [Entrypoint] Starting metadata enrichment for: semantic-rag-single_table_basic-8f7bcf77
2026-03-22 16:17:42,924 - metadata_agent.main - INFO - [MainThread] DynamoDB status updated: semantic-rag-single_table_basic-8f7bcf77 (version=v1) → processing
2026-03-22 16:17:42,926 - metadata_agent.main - INFO - [MainThread] [Entrypoint] 1 tables to process


{"timestamp": "2026-03-22T20:17:42.926Z", "level": "INFO", "message": "Async task started: metadata_enrichment (ID: -2764093469475915705)", "logger": "bedrock_agentcore.app"}


2026-03-22 16:17:42,926 - bedrock_agentcore.app - INFO - [MainThread] Async task started: metadata_enrichment (ID: -2764093469475915705)
2026-03-22 16:17:42,927 - metadata_agent.main - INFO - [metadata-enrichment] [Revision] Revision mode for semantic-rag-single_table_basic-8f7bcf77
Polling v2 for completion (max 600s)...2026-03-22 16:17:42,928 - metadata_agent.main - INFO - [metadata-enrichment] [Revision] [Table 1/1] normalized.admin_codes (catalog: s3tablescatalog/semantic-layer-analytics-tables)



I'll process the annotation hints for the `admin_codes` table. Let me start by fetching the existing schema.
Tool #1: get_single_table_schema
2026-03-22 16:17:46,574 - metadata_agent.main - INFO - [asyncio_0] get_single_table_schema called: database='normalized', table='admin_codes', catalog_id='s3tablescatalog/semantic-layer-analytics-tables'
2026-03-22 16:17:46,575 - metadata_agent.main - INFO - [asyncio_0] Routing 'normalized.admin_codes': S3 Tables (Glue direct)
2026-03-22 16:17:46,5

{"timestamp": "2026-03-22T20:18:15.781Z", "level": "INFO", "message": "Async task completed: metadata_enrichment (ID: -2764093469475915705, Duration: 32.85s)", "logger": "bedrock_agentcore.app"}


2026-03-22 16:18:15,781 - bedrock_agentcore.app - INFO - [metadata-enrichment] Async task completed: metadata_enrichment (ID: -2764093469475915705, Duration: 32.85s)
  [45s] v2 status=completed

✓ Final status: completed
  version: v2
  summary: Processed 1/1 tables.


## Summary

### Agent Workflow
The Metadata Generation Agent enriches the AWS Glue Data Catalog and Bedrock Knowledge Base:

1. **`get_single_table_schema(database_name, table_name, catalog_id)`** — fetches column definitions via Glue directly for S3 Tables (Iceberg) catalogs, or Athena DESCRIBE TABLE for standard Glue tables (with Glue fallback for DynamoDB-backed tables)
2. **`sample_table_data(database_name, table_name, catalog_id, sample_size=10)`** — retrieves example rows to ground descriptions in real data; falls back to a DynamoDB Scan for DynamoDB-backed tables
3. **`update_glue_table_metadata(database_name, table_name, table_description, column_descriptions, catalog_id="")`** — writes the table description and per-column comments back to Glue; pass `'s3tablescatalog/<bucket>'` for S3 Tables (Iceberg)
4. **`save_metadata_document_to_s3(database_name, table_name, catalog_id, metadata_content)`** — persists a structured markdown document to S3 for Bedrock KB ingestion; also writes a companion `.metadata.json` file with `database_name`, `table_name`, and `catalog_id` as Bedrock KB chunk attributes
5. **`update_progress(id, tables_processed, total_tables, current_table)`** — records enrichment progress in DynamoDB so callers can poll for status

### Multi-Dimensional Evaluation
- **HelpfulnessEvaluator**: Description quality and business clarity (7-level scale)
- **FaithfulnessEvaluator**: Accuracy relative to retrieved schema/sample data — detects hallucinations (5-level scale)
- **ToolSelectionAccuracyEvaluator**: Correct tool usage and ordering (schema → sample → Glue write → S3 save → progress)
- **GoalSuccessRateEvaluator**: End-to-end enrichment success rate

### Metadata-Specific Validation Criteria
1. **Schema Coverage**: Every column in `get_single_table_schema` output has a description in `update_glue_table_metadata`
2. **Comment Length**: All column comments ≤ 255 chars (Glue limit)
3. **S3 Document Structure**: Markdown contains `## Overview`, `## Columns`, `## Sample Data`, `## Notes`
4. **Business Language**: Descriptions explain meaning, not just restate column names
5. **FK Awareness**: Foreign key columns reference the correct target entity

### Next Steps
- Review failed test cases to understand agent weaknesses
- Expand to multi-table test cases
- Verify S3 documents are correctly indexed by Bedrock KB after ingestion
- Track description quality metrics over time as the agent prompt evolves